In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (folder for folder in (start, *start.parents)
     if (folder / "replication_paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Start Jupyter from the replication project or a subfolder of it. "
        "The project root must contain replication_paths.py."
    )
for source_dir in (PROJECT_ROOT, PROJECT_ROOT / "01_taxonomy", PROJECT_ROOT / "03_judge_prompts",
                   PROJECT_ROOT / "04_llm_evaluation"):
    if str(source_dir) not in sys.path:
        sys.path.insert(0, str(source_dir))
from replication_paths import DATASET_DIR, RAW_DEVGPT_DIR, INTERIM_DIR, RESULTS_DIR, HUMAN_FILE, OUTPUT_DIR

from itertools import combinations
import json
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

input_dir = RESULTS_DIR
output_dir = OUTPUT_DIR / "cooccurrence"
models = ["gpt", "claude", "gemini", "deepseek"]
techniques = ["zero_shot", "few_shot", "mv3"]

In [ ]:

smells = [
    "grammar_smell", "formatting_smell", "readability_smell",
    "ambiguous_instruction", "missing_constraints", "conflicting_instructions",
    "intent_misalignment", "missing_context_smell", "irrelevant_context",
    "complexity_smell", "excessive_prompt_length", "inefficient_few_shot_examples",
    "overloaded_prompt", "undefined_output_format", "poor_prompt_organization",
    "role_separation_smell", "incorrect_information", "prompt_injection",
    "unsafe_prompt", "bias_smell", "style_formality_mismatch",
]


In [ ]:
##load classifications

rows = []

for model in models:
    for technique in techniques:
        filename = "classifications.jsonl" if technique == "mv3" else "responses.jsonl"
        path = input_dir / f"{model}_{technique}" / filename

        for line in path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                continue

            data = json.loads(line)
            result = data.get("result") or data
            prompt_id = result.get("prompt_id") or data.get("prompt_id")
            values = result.get("smells")

            if prompt_id and isinstance(values, dict):
                rows.append((model, technique, str(prompt_id), values))

df = pd.DataFrame(rows, columns=["model", "technique", "prompt_id", "smells"])

keys = ["model", "technique", "prompt_id"]

if df.empty or df.duplicated(keys).any():
    raise ValueError("Empty dataset or duplicated prompt IDs")
if df.groupby(["model", "technique"]).ngroups != len(models) * len(techniques):
    raise ValueError("Some model/technique configurations have no valid data")

In [ ]:
def binary_label(row, smell):
    item = row.smells.get(smell)

    if item is None:
        return 0.0

    label = item.get("label") if isinstance(item, dict) else item
    label = label.strip().lower() if isinstance(label, str) else label

    return {"present": 1.0, "absent": 0.0}.get(label, np.nan)


binary = df[keys].copy()

for smell in smells:
    binary[smell] = df.apply(
        lambda row: binary_label(row, smell), axis=1
    )

In [ ]:
def pair_metrics(group, a, b):
    pair = group[[a, b]].dropna()
    ##a and b are each prompt smell being evaluated
    n = len(pair)
    ##n is number of prompts with valid classifications for both smells.
    if not n:
        return None
    ## na: number of prompts containing smell a and nb containing b and nab both
    na, nb = int(pair[a].sum()), int(pair[b].sum())
    nab = int(((pair[a] == 1) & (pair[b] == 1)).sum())
    ## calculates the expected number of prompts containing both smells if they were statistically independent.
    expected = na * nb / n
    ##Fisher's exact test evaluates whether the observed positive association is stronger than expected under independence
    p = fisher_exact([[nab, na - nab], [nb - nab, n - na - nb + nab]],
                     alternative="greater").pvalue
    
    ##calculate support, confidence, and lift
    return {
        "n_valid": n, "n_ab": nab, "support": nab / n,
        "confidence_a_to_b": nab / na if na else np.nan,
        "confidence_b_to_a": nab / nb if nb else np.nan,
        "lift": nab / expected if expected else np.nan,
        "p_value": p,
    }

In [ ]:
results = []
## every possible pair of smells, separately for each model and technique
for (model, technique), group in binary.groupby(["model", "technique"]):
    for a, b in combinations(smells, 2):
        metrics = pair_metrics(group, a, b)
        if metrics is not None:
            results.append({"model": model, "technique": technique,
                            "smell_a": a, "smell_b": b, **metrics})

pairs = pd.DataFrame(results)
if pairs.empty:
    raise ValueError("No valid smell pairs found")
pairs["q_value"] = pairs.groupby(["model", "technique"])["p_value"].transform(
    lambda values: multipletests(values, method="fdr_bh")[1]
)
display(
    pairs.sort_values("lift", ascending=False)
         .head(20)
         .round(3)
)

In [ ]:
summary = pairs.dropna(subset=["lift"]).copy()

summary["positive"] = summary["lift"] > 1
summary["significant"] = summary["positive"] & (summary["q_value"] < 0.05)

summary = summary.groupby(["smell_a", "smell_b"], as_index=False).agg(
    configs=("lift", "size"),
    positive_configs=("positive", "sum"),
    significant_configs=("significant", "sum"),
    support=("support", "mean"),
    confidence_a_to_b=("confidence_a_to_b", "mean"),
    confidence_b_to_a=("confidence_b_to_a", "mean"),
    lift=("lift", "mean"),
)

summary["positive_share"] = summary["positive_configs"] / summary["configs"]

summary = summary.sort_values(
    ["positive_configs", "significant_configs", "lift"],
    ascending=False
)

display(summary.head(20).round(3))

In [ ]:
display(
    pairs[
        (pairs.smell_a == "grammar_smell") &
        (pairs.smell_b == "formatting_smell")
    ][["model", "technique", "n_valid", "n_ab",
       "support", "lift", "q_value"]].round(3)
)

In [ ]:
primary = pairs[pairs["technique"] == "mv3"].copy()

top_pairs = (
    primary.groupby(["smell_a", "smell_b"])["support"]
    .mean()
    .nlargest(15)
    .index
)

In [ ]:
support_table = primary.pivot(
    index=["smell_a", "smell_b"],
    columns="model",
    values="support"
).reindex(top_pairs)

display(support_table.round(3))

In [ ]:
lift_table = primary.pivot(
    index=["smell_a", "smell_b"],
    columns="model",
    values="lift"
).reindex(top_pairs)

display(lift_table.round(3))

In [ ]:
prompt_counts = binary[["model", "technique"]].copy()
prompt_counts["detected_smells"] = binary[smells].eq(1).sum(axis=1)

cooccurrence = (
    prompt_counts.groupby(["model", "technique"])
    .agg(
        prompts_with_smells=("detected_smells", lambda x: (x >= 1).sum()),
        exactly_one_detected=("detected_smells", lambda x: (x == 1).sum()),
        two_or_more_detected=("detected_smells", lambda x: (x >= 2).sum()),
    )
)

cooccurrence["multiple_smells_pct"] = (
    cooccurrence["two_or_more_detected"]
    .div(cooccurrence["prompts_with_smells"].replace(0, float("nan")))
    .mul(100)
)

display(cooccurrence.round(2))

In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)
pairs.to_csv(output_dir / "cooccurrence_by_configuration.csv", index=False)
summary.to_csv(output_dir / "cooccurrence_summary.csv", index=False)